In [2]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

In [10]:
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from openai import OpenAI

from rag_helper import RAGBase

In [11]:
load_dotenv()
openai_client = OpenAI()

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
vs_index = VectorSearchIndex(
    keyword_fields = ["course"],
    mode = "ivf",
    db_path = "faq_vectors2.db"
)

In [6]:
query = "I just discovered the course. Can I still join it?"

In [7]:
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results = 5)

In [8]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [12]:
class RAGVector(RAGBase):
    
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
    
    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}
        
        return self.index.search(
            query_vector,
            filter_dict = filter_dict,
            num_results=num_results
        )
        

In [13]:
vector_assistant = RAGVector(
    embedder = model,
    index = vs_index,
    llm_client = openai_client
)

In [14]:
vector_assistant.rag(query)

'Yes, you can still join. But if you want a certificate, you need to submit your project while the course is still accepting submissions.'